In [0]:
print("123")

In [0]:
!pip install statsmodels scipy lightgbm hierarchicalforecast -q

# County

In [0]:
# ═══════════════════════════════════════════════════════════════
# county_lgbm.py
# LightGBM panel model for county-level quarterly pharma sales
# Includes: synthetic data generation, feature engineering,
#           training, evaluation, SHAP feature importance
# ═══════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import warnings
import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_percentage_error
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
np.random.seed(42)

In [0]:
# ────────────────────────────────────────────────────────────────
# STEP 1 — Synthetic county-level panel data
# Simulates 16 quarters for 50 counties across 5 states.
# Each county has its own baseline, growth rate, and noise level —
# mimicking the real-world variation across small vs large markets.
# External features: rep visit count, HCP density, median income.
# ────────────────────────────────────────────────────────────────
N_COUNTIES = 50
N_QUARTERS = 16
quarters    = pd.date_range("2020-01-01", periods=N_QUARTERS, freq="QS")

states     = ["CA", "TX", "FL", "NY", "IL"]
state_map  = {i: states[i % len(states)] for i in range(N_COUNTIES)}

rows = []
for cid in range(N_COUNTIES):
    baseline    = np.random.uniform(5_000, 40_000)   # county market size
    growth      = np.random.uniform(0.005, 0.025)    # quarterly growth rate
    noise_scale = baseline * 0.07

    for t, dt in enumerate(quarters):
        trend    = baseline * (1 + growth) ** t
        seasonal = trend * np.array([0.02, -0.04, 0.01, 0.06])[t % 4]
        covid    = -trend * 0.15 if t in [1, 2] else 0   # Q2-Q3 2020 shock

        # External features — correlated with sales but with noise
        rep_visits  = int(np.random.poisson(lam=8 + baseline / 8_000))
        hcp_density = round(np.random.uniform(2.0, 12.0), 1)   # HCPs per 10k pop
        med_income  = round(np.random.uniform(40_000, 90_000), -3)

        trx = trend + seasonal + covid + np.random.normal(0, noise_scale)

        rows.append({
            "county_id":   f"CTY_{cid:03d}",
            "state":       state_map[cid],
            "ds":          dt,
            "trx":         max(trx, 0),
            "rep_visits":  rep_visits,
            "hcp_density": hcp_density,
            "med_income":  med_income,
        })

df = pd.DataFrame(rows)
df["trx"] = df["trx"].round(0)
df.to_csv("base_county.csv", index=False)
df_county_init = df.copy()
df_nation_init = df.copy()
print(f"── Synthetic panel: {df['county_id'].nunique()} counties × {N_QUARTERS} quarters = {len(df)} rows ──")
print(df.groupby("state")["trx"].agg(["mean","sum"]).round(0))

In [0]:
# ────────────────────────────────────────────────────────────────
# STEP 2 — Feature engineering
# LightGBM is a tree model — it cannot extrapolate a trend on its
# own. We must give it trend-capturing features explicitly:
#   • Lag features : what sales were 1, 2, and 4 quarters ago
#   • Rolling stats: mean and std over trailing 4 quarters
#   • Quarter number: captures seasonality (Q4 = 4, etc.)
#   • Time index   : captures the overall trend direction
# Lags introduce NaNs for early rows — we drop those below.
# ────────────────────────────────────────────────────────────────
df = df.sort_values(["county_id", "ds"]).reset_index(drop=True)

for lag in [1, 2, 4]:
    df[f"trx_lag{lag}"] = df.groupby("county_id")["trx"].shift(lag)

df["trx_roll4_mean"] = (
    df.groupby("county_id")["trx"]
    .transform(lambda x: x.shift(1).rolling(4, min_periods=2).mean())
)
df["trx_roll4_std"] = (
    df.groupby("county_id")["trx"]
    .transform(lambda x: x.shift(1).rolling(4, min_periods=2).std().fillna(0))
)

df["quarter_num"] = df["ds"].dt.quarter
df["time_index"]  = df.groupby("county_id").cumcount()   # 0…15 per county

# Encode county_id and state as integers for LightGBM
le_county = LabelEncoder()
le_state  = LabelEncoder()
df["county_encoded"] = le_county.fit_transform(df["county_id"])
df["state_encoded"]  = le_state.fit_transform(df["state"])

print(f"\n── Feature engineering complete. Sample row ──")
print(df[df["county_id"]=="CTY_000"].tail(3).to_string(index=False))

In [0]:
# ────────────────────────────────────────────────────────────────
# STEP 3 — Train / test split
# Temporal split: last 4 quarters = holdout.
# Important: we split by DATE, not randomly, to avoid data leakage.
# A random split would let future lags leak into training rows.
# ────────────────────────────────────────────────────────────────
HOLDOUT_DATE = "2023-01-01"

# Drop rows where lag features are NaN (first 4 quarters per county)
FEATURES = [
    "county_encoded", "state_encoded",
    "quarter_num", "time_index",
    "trx_lag1", "trx_lag2", "trx_lag4",
    "trx_roll4_mean", "trx_roll4_std",
    "rep_visits", "hcp_density", "med_income",
]
df_model = df.dropna(subset=FEATURES).copy()

df_train = df_model[df_model["ds"] <  HOLDOUT_DATE]
df_test  = df_model[df_model["ds"] >= HOLDOUT_DATE]

X_train, y_train = df_train[FEATURES], df_train["trx"]
X_test,  y_test  = df_test[FEATURES],  df_test["trx"]

print(f"\n── Train: {len(df_train)} rows | Test: {len(df_test)} rows ──")

In [0]:
# ────────────────────────────────────────────────────────────────
# STEP 4 — Train LightGBM panel model
# One model learns from ALL counties simultaneously.
# Key hyperparameters for a pharma panel:
#   objective=regression_l1  : MAE loss; robust to occasional large-
#                               volume counties dominating gradients
#   min_child_samples=10     : minimum rows per leaf — prevents the
#                               model from memorizing tiny counties
#   num_leaves=64            : tree complexity; 64 is moderate
#   early_stopping_rounds    : stop when validation MAPE stops falling
# ────────────────────────────────────────────────────────────────
params = {
    "objective":             "regression_l1",
    "metric":                "mape",
    "learning_rate":         0.05,
    "num_leaves":            64,
    "min_child_samples":     10,
    "subsample":             0.8,
    "colsample_bytree":      0.8,
    "n_estimators":          600,
    "early_stopping_rounds": 40,
    "verbose":               -1,
}

model = lgb.LGBMRegressor(**params)
model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[lgb.log_evaluation(period=100)],
)

In [0]:
# ────────────────────────────────────────────────────────────────
# STEP 5 — Evaluate
# We evaluate at two levels:
#   (a) Overall panel MAPE
#   (b) Per-county MAPE — important for spotting specific
#       counties where the model systematically under/over-forecasts
# ────────────────────────────────────────────────────────────────
y_pred = model.predict(X_test).clip(min=0)

overall_mape = mean_absolute_percentage_error(y_test, y_pred)
print(f"\n── Overall holdout MAPE: {overall_mape:.1%} ──")

eval_df = df_test[["county_id","state","ds","trx"]].copy()
eval_df["forecast"]  = y_pred.round(0)
eval_df["pct_error"] = ((y_pred - y_test.values) / (y_test.values + 1e-9) * 100).round(1)

per_county_mape = (
    eval_df.groupby("county_id")
    .apply(lambda g: mean_absolute_percentage_error(g["trx"], g["forecast"]))
    .reset_index()
    .rename(columns={0: "mape"})
    .sort_values("mape", ascending=False)
)
print("\n── Worst 5 counties by holdout MAPE ──")
print(per_county_mape.head(5).to_string(index=False))
print("\n── Best 5 counties by holdout MAPE ──")
print(per_county_mape.tail(5).to_string(index=False))

In [0]:
# ────────────────────────────────────────────────────────────────
# STEP 6 — Feature importance
# LightGBM's gain-based importance tells us which features drive
# the most reduction in loss at split points across all trees.
# This directly answers: "what makes county sales go up or down?"
# ────────────────────────────────────────────────────────────────
importance = pd.DataFrame({
    "feature":    FEATURES,
    "importance": model.feature_importances_,
}).sort_values("importance", ascending=True)

print("\n── Feature importance (gain) ──")
print(importance.sort_values("importance", ascending=False).to_string(index=False))

In [0]:
# ────────────────────────────────────────────────────────────────
# STEP 7 — Plots  (2-panel figure)
#   Left  : actual vs forecast for a sample county (CTY_000)
#   Right : feature importance bar chart
# ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Panel A — sample county forecast
ax = axes[0]
sample = df[df["county_id"] == "CTY_000"].sort_values("ds")
pred_sample = eval_df[eval_df["county_id"] == "CTY_000"].sort_values("ds")

ax.plot(sample["ds"], sample["trx"], color="#378ADD", label="Actuals")
ax.plot(pred_sample["ds"], pred_sample["forecast"],
        color="#D85A30", marker="o", markersize=4, label="Forecast (holdout)")
ax.axvline(pd.Timestamp(HOLDOUT_DATE), color="#888780",
           linestyle="--", linewidth=0.9, label="Train/test split")
ax.set_title("County CTY_000 — actual vs forecast", fontsize=11)
ax.set_ylabel("TRx"); ax.legend(fontsize=8); ax.grid(alpha=0.3)

# Panel B — feature importance
ax2 = axes[1]
colors = ["#378ADD" if i >= len(FEATURES) - 3 else "#B5D4F4"
          for i in range(len(importance))]
ax2.barh(importance["feature"], importance["importance"], color=colors)
ax2.set_title("Feature importance (gain)", fontsize=11)
ax2.set_xlabel("Importance score"); ax2.grid(alpha=0.3, axis="x")

plt.tight_layout()
plt.savefig("plot_county_lgbm.png", dpi=140, bbox_inches="tight")
print("\nPlot saved → plot_county_lgbm.png")

In [0]:
# ────────────────────────────────────────────────────────────────
# STEP 8 — Export for MinT reconciliation
# ────────────────────────────────────────────────────────────────
county_out = eval_df[["county_id","ds","forecast"]].rename(
    columns={"county_id":"geo_id","forecast":"forecast"}
)
county_out["level"] = "county"
df_state_init = county_out.copy()
county_out.to_csv("forecast_county_lgbm.csv", index=False)
print("Forecast csv saved → forecast_county_lgbm.csv")

In [0]:
county_out

# State

## State Data Prep

In [0]:
## only the forecast values
df_state_init

In [0]:
geo_walk = pd.read_csv("geo_crosswalk.csv")
geo_walk

In [0]:
df_state = pd.merge(df_state_init, geo_walk, left_on="geo_id", right_on = "county_fips", how="left")
df_state.drop(["level", "zip_code", "geo_id"], axis=1, inplace=True)
df_state.drop_duplicates(inplace=True)
df_state

In [0]:
df_state_base = df_state.groupby(["state_abbr", "ds"])["forecast"].sum().reset_index()
df_state_base.head()

In [0]:
df_state_delta = pd.read_csv("delta_state.csv")
df_state_delta['ds'] = pd.to_datetime(df_state_delta['ds'])
df_state = pd.merge(df_state_base, df_state_delta, on = ["state_abbr", "ds"])
df_state["trx"] = df_state["forecast"] + df_state["delta"]
df_state.drop(["forecast", "delta"], axis=1, inplace=True)
# df_state.drop_duplicates(inplace=True)
df_state

## State output

In [0]:
state_out = df_state.rename(
    columns={
        "state_abbr": "geo_id",   # old name → new name
        "trx": "forecast"
    }
)
state_out["level"] = "state"
state_out

# National ARIMA

In [0]:
# ═══════════════════════════════════════════════════════════════
# national_arima.py
# ARIMA-X model for national-level quarterly pharma sales
# Includes: synthetic data generation, model fitting, evaluation
# ═══════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import warnings
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
np.random.seed(42)

In [0]:
# ────────────────────────────────────────────────────────────────
# STEP 1 — Synthetic national data
# Simulates 16 quarters (Q1 2020 – Q4 2023) of national TRx sales
# Components:
#   • Upward trend (~+2% per quarter, organic growth)
#   • Quarterly seasonality (Q4 flu season bump, Q2 dip)
#   • A COVID shock in 2020 Q2-Q3 (HCP visits dropped)
#   • Promotional spend as an external regressor
#   • Gaussian noise
# ────────────────────────────────────────────────────────────────
quarters = pd.date_range("2020-01-01", periods=16, freq="QS")
n = len(quarters)

trend      = np.linspace(100_000, 135_000, n)
seasonality = np.tile([0.02, -0.04, 0.01, 0.06], 4)     # Q1-Q4 multipliers
covid_shock = np.array([0,-0.18,-0.10,0,0,0,0,0,0,0,0,0,0,0,0,0])  # Q2-Q3 2020

promo_spend = np.array([
    1.0,0.8,1.1,1.3, 1.0,0.9,1.2,1.4,
    1.1,1.0,1.3,1.5, 1.2,1.1,1.4,1.6
])  # normalized spend index; >1 = above baseline

noise = np.random.normal(0, 2_000, n)

trx = trend * (1 + seasonality + covid_shock) + promo_spend * 3_000 + noise
trx = trx.clip(min=0)

df_nat = pd.DataFrame({
    "ds":          quarters,
    "trx_delta":         trx.round(0),
    "promo_spend": promo_spend,
    "covid_flag":  (covid_shock != 0).astype(int),
})

print("── Synthetic national data (16 quarters) ──")
print(df_nat.to_string(index=False))

In [0]:
df_nation_init

In [0]:
df_nation_init = df_nation_init[["ds","trx"]]
df_nation_base = df_nation_init.groupby("ds")["trx"].sum().reset_index()
df_nation_base['ds'] = pd.to_datetime(df_nation_base['ds'])
df_nation_base = df_nation_base.rename(columns={"trx":"trx_base"})
df_nation_base

In [0]:
df_national = pd.merge(df_nat, df_nation_base, on = ["ds"])
df_national["trx"] = df_national["trx_base"] + df_national["trx_delta"]
df_national.drop(["trx_base", "trx_delta"], axis=1, inplace=True)
# df_national.drop_duplicates(inplace=True)
df_national

In [0]:
# ────────────────────────────────────────────────────────────────
# STEP 2 — Stationarity check (ADF test)
# ARIMA requires a stationary series (constant mean & variance).
# If the series is non-stationary we difference it (d=1).
# ADF null hypothesis: series has a unit root (non-stationary).
# p < 0.05 → reject null → series IS stationary.
# ────────────────────────────────────────────────────────────────
adf_result = adfuller(df_national["trx"], autolag="AIC")
print(f"\n── ADF test (raw TRx) ──")
print(f"  ADF statistic : {adf_result[0]:.4f}")
print(f"  p-value       : {adf_result[1]:.4f}")
print(f"  Conclusion    : {'Stationary' if adf_result[1] < 0.05 else 'Non-stationary → will use d=1'}")

In [0]:
# ────────────────────────────────────────────────────────────────
# STEP 3 — Train / test split
# Hold out last 4 quarters as the validation window.
# This mirrors real deployment: train on Q1-2020 → Q4-2022,
# forecast Q1-2023 → Q4-2023, compare against known actuals.
# ────────────────────────────────────────────────────────────────
HOLDOUT = 4
df_train = df_national.iloc[:-HOLDOUT].copy()
df_test  = df_national.iloc[-HOLDOUT:].copy()

y_train   = df_train["trx"]
exog_train = df_train[["promo_spend", "covid_flag"]]
exog_test  = df_test[["promo_spend", "covid_flag"]]

In [0]:
# ────────────────────────────────────────────────────────────────
# STEP 4 — Fit SARIMAX model
# Order (p, d, q):
#   p=1  — one autoregressive lag (TRx depends on previous quarter)
#   d=1  — first-order differencing to remove trend non-stationarity
#   q=1  — one moving average term (smooths short-term shocks)
# Seasonal order (P, D, Q, s) with s=4 (quarterly):
#   P=1, D=0, Q=1 — mild seasonal AR and MA terms
# exog — include promo_spend and covid_flag as external regressors
# ────────────────────────────────────────────────────────────────
model = SARIMAX(
    endog=y_train,
    exog=exog_train,
    order=(1, 1, 1),               # (p, d, q)
    seasonal_order=(1, 0, 1, 4),   # (P, D, Q, s)
    enforce_stationarity=False,
    enforce_invertibility=False,
)

result = model.fit(disp=False)

print("\n── SARIMAX model summary ──")
print(result.summary().tables[1])  # coefficient table only

In [0]:
# ────────────────────────────────────────────────────────────────
# STEP 5 — Forecast on holdout
# get_forecast() returns point estimates + confidence intervals.
# We pass exog_test so the model can use known promo/covid values
# for the 4 forecast quarters.
# ────────────────────────────────────────────────────────────────
forecast_obj   = result.get_forecast(steps=HOLDOUT, exog=exog_test)
forecast_mean  = forecast_obj.predicted_mean
forecast_ci    = forecast_obj.conf_int(alpha=0.20)   # 80% CI

forecast_mean  = forecast_mean.clip(lower=0)

actuals   = df_test["trx"].values
predicted = forecast_mean.values
mape = np.mean(np.abs((actuals - predicted) / (actuals + 1e-9))) * 100

print(f"\n── Holdout results (last {HOLDOUT} quarters) ──")
results_df = pd.DataFrame({
    "quarter":   df_test["ds"].dt.to_period("Q").astype(str),
    "actual":    actuals.round(0),
    "forecast":  predicted.round(0),
    "pct_error": ((predicted - actuals) / actuals * 100).round(1),
    "lo_80":     forecast_ci.iloc[:, 0].clip(0).round(0).values,
    "hi_80":     forecast_ci.iloc[:, 1].round(0).values,
})
print(results_df.to_string(index=False))
print(f"\n  MAPE: {mape:.1f}%")


In [0]:
# ────────────────────────────────────────────────────────────────
# STEP 6 — Diagnostic plots
# Saves two plots:
#   (a) Forecast vs actuals with confidence band
#   (b) Residual diagnostics (for model validation)
# ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Panel A — forecast vs actuals
ax = axes[0]
ax.plot(df_train["ds"], df_train["trx"], color="#378ADD", label="Train actuals")
ax.plot(df_test["ds"],  df_test["trx"],  color="#378ADD", linestyle="--", label="Test actuals")
ax.plot(df_test["ds"],  predicted,       color="#D85A30", label="Forecast")
ax.fill_between(
    df_test["ds"],
    forecast_ci.iloc[:, 0].clip(0),
    forecast_ci.iloc[:, 1],
    color="#D85A30", alpha=0.15, label="80% CI"
)
ax.set_title("National TRx — ARIMA forecast vs actuals", fontsize=11)
ax.set_ylabel("TRx"); ax.legend(fontsize=8); ax.grid(alpha=0.3)

# Panel B — residuals
residuals = result.resid
ax2 = axes[1]
ax2.plot(df_train["ds"], residuals, color="#888780")
ax2.axhline(0, color="#E24B4A", linewidth=0.8)
ax2.set_title("Model residuals (training period)", fontsize=11)
ax2.set_ylabel("Residual"); ax2.grid(alpha=0.3)

plt.tight_layout()
# plt.show()
plt.savefig("plot_national_arima.png", dpi=140, bbox_inches="tight")
print("\nPlot saved → plot_national_arima.png")

In [0]:
# ────────────────────────────────────────────────────────────────
# STEP 7 — Export for MinT reconciliation
# Standard schema: geo_id | ds | forecast | level
# ────────────────────────────────────────────────────────────────
national_out = pd.DataFrame({
    "geo_id":   "USA",
    "ds":       df_test["ds"],
    "forecast": predicted.round(0),
    "lo_80":    forecast_ci.iloc[:, 0].clip(0).round(0).values,
    "hi_80":    forecast_ci.iloc[:, 1].round(0).values,
    "level":    "national",
})
national_out.to_csv("forecast_national_arima.csv", index=False)
print("Forecast csv saved → forecast_national_arima.csv")

In [0]:
national_out

# ZIP

In [0]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_percentage_error

In [0]:
delta_zip = pd.read_csv("delta_zip.csv")
delta_zip

In [0]:
# Unpivot (melt) the DataFrame
df_delta_zip = pd.melt(
    delta_zip,
    id_vars=["zip_code"],          # columns to keep fixed
    var_name="ds",           # new column name for former headers
    value_name="delta_zip"       # new column name for values
)

df_delta_zip

In [0]:
df_county_base = df_county_init[["county_id", "ds", "trx"]]
df_county_mean = (
    df_county_base
    .groupby(["county_id"])["trx"]
    .mean()
    .reset_index()  # converts group keys back to columns
)
df_county_mean.head()

In [0]:
df_zip_base = geo_walk.merge(df_county_mean, left_on = "county_fips", right_on="county_id", how="left")
df_zip_base = df_zip_base.merge(df_delta_zip, on="zip_code", how="left")
df_zip_base["trx_with_delta"] = df_zip_base["trx"].fillna(0) + df_zip_base["delta_zip"].fillna(0)
df_zip_base

In [0]:
df = df_zip_base[["zip_code", "ds", "trx_with_delta"]]
df = df.rename(columns={"trx_with_delta": "trx"})
actuals_zip = df.copy()
df 

## Next Steps

In [0]:
# ── 1. Load panel data ───────────────────────────────────────
# Expected schema: zip_code, quarter_end_date, trx, + feature cols
df["ds"] = pd.to_datetime(df["ds"])
df = df.sort_values(["zip_code", "ds"]).reset_index(drop=True)

In [0]:
# ── 2. Route sparse ZIPs ─────────────────────────────────────
def classify_zip(series: pd.Series) -> str:
    """
    Dense   : >= 10 non-zero quarters  → LightGBM panel
    Sparse  : 4–9 non-zero quarters   → Croston's method
    Dead    : < 4 non-zero quarters   → propagate from county forecast
    """
    nonzero = (series > 0).sum()
    if nonzero >= 10:
        return "dense"
    elif nonzero >= 4:
        return "sparse"
    else:
        return "dead"

zip_class = (
    df.groupby("zip_code")["trx"]
    .apply(classify_zip)
    .reset_index()
    .rename(columns={"trx": "zip_class"})
)
df = df.merge(zip_class, on="zip_code")

print(df["zip_class"].value_counts())   # sanity check routing split

df_dense  = df[df["zip_class"] == "dense"].copy()
df_sparse = df[df["zip_class"] == "sparse"].copy()

In [0]:
# ── 3. Feature engineering (dense ZIPs) ─────────────────────
def make_features(panel: pd.DataFrame) -> pd.DataFrame:
    panel = panel.sort_values(["zip_code", "ds"])

    # Lag features — previous quarters' sales
    for lag in [1, 2, 4]:
        panel[f"trx_lag{lag}"] = (
            panel.groupby("zip_code")["trx"].shift(lag)
        )

    # Rolling statistics
    panel["trx_roll4_mean"] = (
        panel.groupby("zip_code")["trx"]
        .transform(lambda x: x.shift(1).rolling(4).mean())
    )
    panel["trx_roll4_std"] = (
        panel.groupby("zip_code")["trx"]
        .transform(lambda x: x.shift(1).rolling(4).std())
    )

    # Quarter-of-year (seasonality signal)
    panel["quarter_num"] = panel["ds"].dt.quarter

    # Encode ZIP as integer (LightGBM handles categorical natively)
    le = LabelEncoder()
    panel["zip_encoded"] = le.fit_transform(panel["zip_code"])

    return panel, le

df_dense, zip_encoder = make_features(df_dense)

In [0]:
# ── 4. Define feature set ────────────────────────────────────
FEATURES = [
    "zip_encoded",
    "quarter_num",
    "trx_lag1", "trx_lag2", "trx_lag4",
    "trx_roll4_mean", "trx_roll4_std",
    # add external regressors here if available at ZIP level:
    # "rep_visit_count", "hcp_count_in_zip", "median_hh_income",
]
TARGET = "trx"

# Drop rows with NaN lags (first few periods per ZIP)
df_model = df_dense.dropna(subset=FEATURES)

HOLDOUT_QTR = "2023-10-01"   # last 4 quarters = holdout
df_train = df_model[df_model["ds"] < HOLDOUT_QTR]
df_test  = df_model[df_model["ds"] >= HOLDOUT_QTR]

X_train, y_train = df_train[FEATURES], df_train[TARGET]
X_test,  y_test  = df_test[FEATURES],  df_test[TARGET]

In [0]:
y_test.head()

In [0]:
# ── 5. Train LightGBM panel model ───────────────────────────
params = {
    "objective":        "regression_l1",   # MAE loss; robust to outliers
    "metric":           "mape",
    "learning_rate":    0.05,
    "num_leaves":       64,
    "min_child_samples": 20,               # prevents over-fitting small ZIPs
    "subsample":        0.8,
    "colsample_bytree": 0.8,
    "n_estimators":     500,
    "early_stopping_rounds": 30,
    "verbose":          -1,
}

model_zip = lgb.LGBMRegressor(**params)
model_zip.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
)

mape = mean_absolute_percentage_error(y_test, model_zip.predict(X_test))
print(f"ZIP panel holdout MAPE: {mape:.1%}")

In [0]:
# ── 6. Sparse ZIP handler — Croston's method ─────────────────
def crostons_forecast(series: np.ndarray, horizon: int = 4) -> np.ndarray:
    """
    Classic Croston's method for intermittent demand.
    Separately smooths the non-zero demand and inter-demand intervals.
    Returns a flat forecast array of length `horizon`.
    """
    alpha = 0.1       # smoothing parameter
    demand, interval = [], []
    d_hat, i_hat = None, None
    last_nonzero = None

    for t, val in enumerate(series):
        if val > 0:
            if d_hat is None:
                d_hat = val
                i_hat = 1 if last_nonzero is None else t - last_nonzero
            else:
                d_hat = alpha * val + (1 - alpha) * d_hat
                gap   = t - last_nonzero
                i_hat = alpha * gap + (1 - alpha) * i_hat
            last_nonzero = t

    if d_hat is None or i_hat is None or i_hat == 0:
        return np.zeros(horizon)

    rate = d_hat / i_hat   # expected demand per period
    return np.full(horizon, rate)


sparse_forecasts = {}
for zip_code, grp in df_sparse.groupby("zip_code"):
    series = grp.sort_values("ds")["trx"].values
    sparse_forecasts[zip_code] = crostons_forecast(series, horizon=4)


In [0]:
# ── 7. Assemble ZIP-level forecast output ────────────────────
HORIZON_DATES = pd.date_range("2024-01-01", periods=4, freq="QS")

# Dense ZIPs — predict on constructed future feature rows
# (In production: build a future feature dataframe the same way as training)
# Here we show the structure; replace with actual future feature rows
zip_forecast_rows = []
for zip_code in df_dense["zip_code"].unique():
    preds = model_zip.predict(X_test[X_test.index.isin(
        df_test[df_test["zip_code"] == zip_code].index
    )])
    for dt, pred in zip(HORIZON_DATES[:len(preds)], preds):
        zip_forecast_rows.append({
            "geo_id":   zip_code,
            "ds":       dt,
            "forecast": max(pred, 0),   # floor at 0 — no negative Rx
            "level":    "zip",
            "method":   "lgbm_panel",
        })

# Sparse ZIPs
for zip_code, preds in sparse_forecasts.items():
    for dt, pred in zip(HORIZON_DATES, preds):
        zip_forecast_rows.append({
            "geo_id":   zip_code,
            "ds":       dt,
            "forecast": float(pred),
            "level":    "zip",
            "method":   "crostons",
        })

zip_out = pd.DataFrame(zip_forecast_rows)
zip_out.to_csv("forecast_zips.csv", index=False)
print(f"ZIP forecasts written: {len(zip_out)} rows")


In [0]:
zip_out

# MinT

In [0]:
# ─────────────────────────────────────────────────────────────
# mint_reconciliation.py
# Hierarchical forecast reconciliation using MinT (Minimum Trace)
# Ensures: ZIP forecasts sum → county → state → national
#
# Uses the `hierarchicalforecast` library by Nixtla
#   pip install hierarchicalforecast
#
# Input:  base forecasts from national_model.py + zip_model.py
#         + state & county base forecasts (same schema)
# Output: reconciled forecasts at all geo levels
# ─────────────────────────────────────────────────────────────

import pandas as pd
import numpy as np
from hierarchicalforecast.core import HierarchicalReconciliation
from hierarchicalforecast.methods import MinTrace
from hierarchicalforecast.utils import aggregate

In [0]:
# ── 1. Load base forecasts from all levels ───────────────────
# Schema for each: geo_id | ds | forecast | level
# nat_fc    = pd.read_parquet("data/forecasts/national_base.parquet")
# state_fc  = pd.read_parquet("data/forecasts/state_base.parquet")
# county_fc = pd.read_parquet("data/forecasts/county_base.parquet")
# zip_fc    = pd.read_parquet("data/forecasts/zip_base.parquet")

In [0]:
nat_fc = national_out
state_fc = state_out
county_fc = county_out
zip_fc = zip_out

In [0]:
# import pandas as pd

# nat_fc    = pd.read_csv("nat_b4.csv")
# state_fc  = pd.read_csv("state_b4.csv")
# county_fc = pd.read_csv("county_lgbm_forecast.csv")
# zip_fc    = pd.read_csv("data/forecasts/zip_base.csv")

In [0]:
# ── 2. Build the hierarchy spec ──────────────────────────────
# geo_crosswalk maps every ZIP to its parent county, state, national
geo_xwalk = pd.read_csv("geo_crosswalk.csv")
# Required columns: zip_code, county_fips, state_abbr
# (national is the single root — we add it as a constant column)
geo_xwalk["national"] = "USA"

# hierarchicalforecast expects a list-of-lists defining each level
# Format: [[level_0_col], [level_1_col], ..., [leaf_col]]
# Each inner list = the columns that uniquely identify a node at that level
HIERARCHY_SPEC = [
    ["national"],
    ["national", "state_abbr"],
    ["national", "state_abbr", "county_fips"],
    ["national", "state_abbr", "county_fips", "zip_code"],
]

In [0]:
# ── 3. Build the wide panel of actuals for covariance estimation ─
# MinT needs historical actuals (not just forecasts) to estimate
# error covariance across the hierarchy
# actuals_zip = pd.read_parquet("data/zip_quarterly_panel.parquet")  # zip_code | ds | trx

# Aggregate actuals up the hierarchy using the crosswalk
actuals_zip = actuals_zip.merge(
    geo_xwalk[["zip_code", "county_fips", "state_abbr", "national"]],
    on="zip_code", how="left"
)
actuals_zip = actuals_zip.rename(columns={"trx": "y"})
actuals_zip

In [0]:
# actuals_zip.to_csv("actuals_zip_out.csv")

In [0]:
# print(set(HIERARCHY_SPEC[-1]["columns"]) - set(actuals_zip.columns))
HIERARCHY_SPEC

In [0]:
# Use hierarchicalforecast's `aggregate` utility to build
# a coherent wide panel: one row per (ds, unique_id), where
# unique_id encodes the full path (e.g. "USA/CA/06037/90210")
Y_df, S_df, tags = aggregate(
    df=actuals_zip,
    spec=HIERARCHY_SPEC,
)
# Y_df  : long-format actuals (unique_id | ds | y)
# S_df  : summing matrix — maps leaf nodes to all aggregates
# tags  : dict mapping level name → list of unique_ids at that level

print(f"Hierarchy nodes: {S_df.shape[1]} bottom, {S_df.shape[0]} total")
print("Level counts:")
for level, ids in tags.items():
    print(f"  {level:15s}: {len(ids):6,} nodes")

In [0]:
# ── 4. Stack base forecasts into the required format ─────────
# hierarchicalforecast expects: unique_id | ds | ModelName
# unique_id must match the format produced by `aggregate` above
# (full path string, e.g. "USA/CA/06037/90210")

def build_unique_id(df: pd.DataFrame, geo_col: str, level: str) -> pd.DataFrame:
    """Attach full-path unique_id from the crosswalk."""
    path_cols = {
        "national": ["national"],
        "state":    ["national", "state_abbr"],
        "county":   ["national", "state_abbr", "county_fips"],
        "zip":      ["national", "state_abbr", "county_fips", "zip_code"],
    }
    df = df.merge(geo_xwalk, on=geo_col, how="left")
    cols = path_cols[level]
    df["unique_id"] = df[cols].apply(lambda r: "/".join(r.astype(str)), axis=1)
    return df[["unique_id", "ds", "forecast"]]

nat_fc_fmt    = build_unique_id(nat_fc.rename(columns={"geo_id":"national"}),    "national",    "national")
state_fc_fmt  = build_unique_id(state_fc.rename(columns={"geo_id":"state_abbr"}),  "state_abbr",  "state")
county_fc_fmt = build_unique_id(county_fc.rename(columns={"geo_id":"county_fips"}),"county_fips", "county")
zip_fc_fmt    = build_unique_id(zip_fc.rename(columns={"geo_id":"zip_code"}),    "zip_code",    "zip")

# Combine all levels; rename 'forecast' → model name column
all_base = pd.concat([nat_fc_fmt, state_fc_fmt, county_fc_fmt, zip_fc_fmt])
all_base = all_base.rename(columns={"forecast": "BaseModel"})

# Pivot to wide: one column per model (here we have one: BaseModel)
Y_hat_df = all_base.pivot_table(
    index=["unique_id", "ds"],
    values="BaseModel",
    aggfunc="first"
).reset_index()

In [0]:
# ── 5. Run MinT reconciliation ───────────────────────────────
hrec = HierarchicalReconciliation(
    reconcilers=[
        MinTrace(method="mint_shrink"),   # shrinkage estimator — best for
                                           # large sparse hierarchies like ZIPs
        # Alternatives to try:
        # MinTrace(method="ols")          — ordinary least squares, faster
        # MinTrace(method="wls_struct")   — weighted by node size
        # BottomUp()                      — simple aggregation, no top signal
    ]
)

reconciled = hrec.reconcile(
    Y_hat_df=Y_hat_df,   # base forecasts, wide format
    Y_df=Y_df,           # historical actuals (for covariance estimation)
    S_df=S_df,              # summing matrix
    tags=tags,           # level metadata
)

# reconciled columns: unique_id | ds | BaseModel | BaseModel/MinTrace_mint_shrink
RECONCILED_COL = "BaseModel/MinTrace_mint_shrink"

In [0]:
# ── 6. Post-process & validate coherence ─────────────────────
# Floor negative reconciled values (can happen in very sparse ZIPs)
reconciled[RECONCILED_COL] = reconciled[RECONCILED_COL].clip(lower=0)

def check_coherence(df: pd.DataFrame, col: str, tol: float = 0.01) -> None:
    """
    Spot-check that ZIP forecasts sum to their parent county.
    Prints any violations above `tol` relative error.
    """
    df = df.copy()
    df["county_id"] = df["unique_id"].apply(lambda x: "/".join(x.split("/")[:3]))
    df["is_zip"]    = df["unique_id"].str.count("/") == 3

    zip_sum    = df[df["is_zip"]].groupby(["county_id", "ds"])[col].sum()
    county_val = df[~df["is_zip"] & (df["unique_id"].str.count("/") == 2)]\
                    .set_index(["unique_id", "ds"])[col]

    for (cid, ds), z_sum in zip_sum.items():
        try:
            c_val = county_val.loc[(cid, ds)]
            rel_err = abs(z_sum - c_val) / (c_val + 1e-9)
            if rel_err > tol:
                print(f"  WARN coherence gap: {cid} {ds} — ZIP sum={z_sum:.1f}, county={c_val:.1f}")
        except KeyError:
            pass

print("\nCoherence check (ZIP → county):")
check_coherence(reconciled, RECONCILED_COL)
print("  Done — no output means all within tolerance.")

In [0]:
# ── 7. Reshape to long output format ─────────────────────────
def extract_level(unique_id: str) -> str:
    depth = unique_id.count("/")
    return {0: "national", 1: "state", 2: "county", 3: "zip"}.get(depth, "unknown")

final = reconciled[["unique_id", "ds", RECONCILED_COL]].copy()
final = final.rename(columns={RECONCILED_COL: "reconciled_forecast"})
final["level"] = final["unique_id"].apply(extract_level)
final["geo_id"] = final["unique_id"].apply(lambda x: x.split("/")[-1])

# Summary
print("\nReconciled forecast counts by level:")
print(final["level"].value_counts())

final.to_parquet("data/forecasts/reconciled_all_levels.parquet", index=False)
print("\nOutput saved → data/forecasts/reconciled_all_levels.parquet")
print(final.head(10).to_string(index=False))